# Cats & Dogs bootstrap (CIFAR-10)

CIFAR-10 is tiny and still available. We grab cat/dog samples from it,
save a minimal folder structure under `data/cats_dogs/`, show a couple of
images, print label counts, and preload tiny text/image embedding models
for the attention experiments. Run from the repo root (ensure `data/` is
writable).

In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image
import torchvision
from torchvision import transforms

DATA_ROOT = Path("data/cats_dogs")
CIFAR_ROOT = Path("data/cifar10")

DATA_ROOT.mkdir(parents=True, exist_ok=True)

transform = transforms.ToTensor()
train_set = torchvision.datasets.CIFAR10(
    root=str(CIFAR_ROOT), train=True, download=True, transform=transform
)
test_set = torchvision.datasets.CIFAR10(
    root=str(CIFAR_ROOT), train=False, download=True, transform=transform
)

cat_label = 3  # 'cat'
dog_label = 5  # 'dog'
label_names = {cat_label: "cat", dog_label: "dog"}


print("done")

In [ ]:
def counts_by_label(ds) -> Counter:
    return Counter(label for _, label in ds)

train_counts = counts_by_label(train_set)
test_counts = counts_by_label(test_set)
print("Train cats/dogs:", {k: train_counts[k] for k in label_names})
print("Test cats/dogs:", {k: test_counts[k] for k in label_names})

def first_k(ds, label, k):
    found = []
    for img, lbl in ds:
        if lbl == label:
            found.append(img)
            if len(found) >= k:
                break
    return found

cats = first_k(train_set, cat_label, 2)
dogs = first_k(train_set, dog_label, 2)

cats_dir = DATA_ROOT / "cats"
dogs_dir = DATA_ROOT / "dogs"
cats_dir.mkdir(parents=True, exist_ok=True)
dogs_dir.mkdir(parents=True, exist_ok=True)

cat_paths: list[Path] = []
dog_paths: list[Path] = []
for i, img in enumerate(cats):
    path = cats_dir / f"cat_{i}.png"
    transforms.ToPILImage()(img).save(path)
    cat_paths.append(path)
for i, img in enumerate(dogs):
    path = dogs_dir / f"dog_{i}.png"
    transforms.ToPILImage()(img).save(path)
    dog_paths.append(path)

print("Saved cat images:", cat_paths)
print("Saved dog images:", dog_paths)

sample_paths = cat_paths + dog_paths


In [ ]:
fig, axes = plt.subplots(1, len(sample_paths), figsize=(10, 3))
for ax, path in zip(axes, sample_paths):
    with Image.open(path) as img:
        ax.imshow(img)
    ax.set_title(path.parent.name)
    ax.axis("off")
plt.tight_layout()


In [ ]:
# Tiny embedding backbones: MobileNetV3-Small (image) + bert-tiny (text)
import torch
from torchvision import models
from transformers import AutoModel, AutoTokenizer

TEXT_MODEL_NAME = "prajjwal1/bert-tiny"
image_weights = models.MobileNet_V3_Small_Weights.DEFAULT
image_model = models.mobilenet_v3_small(weights=image_weights).eval()
image_preprocess = image_weights.transforms()

tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
text_model = AutoModel.from_pretrained(TEXT_MODEL_NAME).eval()

sample_texts = ["a photo of a cat", "a photo of a dog"]
with torch.no_grad():
    text_batch = tokenizer(sample_texts, return_tensors="pt", padding=True, truncation=True)
    text_embeddings = text_model(**text_batch).last_hidden_state[:, 0]
print("Text CLS embeddings shape:", tuple(text_embeddings.shape))

sample_image_path = cat_paths[0]
with Image.open(sample_image_path).convert("RGB") as img:
    img_tensor = image_preprocess(img).unsqueeze(0)
    with torch.no_grad():
        image_features = image_model.features(img_tensor).mean(dim=[2, 3])
print("Image feature shape:", tuple(image_features.shape))


In [ ]:
# === Caption generation from attributes (append this cell at the end) ===
import sys
import subprocess
import json as _json
import csv
from pathlib import Path

# 1) Make sure required packages are installed in this kernel env
def ensure_package(pkg_name: str):
    try:
        __import__(pkg_name)
    except ImportError:
        print(f"[setup] Installing {pkg_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg_name])

ensure_package("torchvision")
ensure_package("transformers")
ensure_package("sentencepiece")

# 2) Imports (after possible install)
import torch
from PIL import Image
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torchvision.transforms as T
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_grad_enabled(False)

# 3) Load segmentation model (multiclass, detailed-ish)
try:
    seg_weights = DeepLabV3_ResNet50_Weights.DEFAULT
    seg_model = deeplabv3_resnet50(weights=seg_weights).to(device).eval()
    seg_preprocess = seg_weights.transforms()
    seg_categories = seg_weights.meta.get("categories", [])
    print("[setup] Loaded DeepLabV3 segmentation model.")
except Exception as e:
    print("[warning] Could not load DeepLabV3 segmentation model, continuing without segmentation.")
    print("          Reason:", e)
    seg_model = None
    seg_preprocess = None
    seg_categories = []

# 4) Load general-purpose image classifier (ImageNet)
clf_weights = ResNet50_Weights.DEFAULT
clf_model = resnet50(weights=clf_weights).to(device).eval()
clf_preprocess = clf_weights.transforms()
clf_categories = clf_weights.meta.get("categories", [])
print("[setup] Loaded ResNet50 classifier.")

# 5) Small local LLM for turning attributes into one sentence
llm_name = "google/flan-t5-small"  # small, reasonable quality
print(f"[setup] Loading LLM: {llm_name} ...")
tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(llm_name).to(device).eval()
print("[setup] LLM loaded.")

def extract_image_attributes(image_path: Path, max_attrs: int = 10):
    """Run segmentation + classification to get attribute strings for an image."""
    img = Image.open(image_path).convert("RGB")
    attrs = []

    # --- segmentation attributes ---
    if seg_model is not None and seg_preprocess is not None and seg_categories:
        img_seg = seg_preprocess(img).unsqueeze(0).to(device)
        with torch.no_grad():
            seg_out = seg_model(img_seg)["out"]  # [1, C, H, W]
            seg_probs = seg_out.softmax(dim=1)[0]  # [C, H, W]
            seg_scores = seg_probs.mean(dim=(1, 2))  # mean score per class

        k = min(5, seg_scores.shape[0])
        topk = torch.topk(seg_scores, k=k)
        seg_attrs = [
            seg_categories[idx]
            for idx, score in zip(topk.indices.tolist(), topk.values.tolist())
            if float(score) > 0.05  # simple threshold
        ]
        attrs.extend(seg_attrs)

    # --- classification attributes ---
    img_clf = clf_preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = clf_model(img_clf)
        probs = logits.softmax(dim=1)[0]
        topk2 = torch.topk(probs, k=5)

    clf_attrs = [clf_categories[idx] for idx in topk2.indices.tolist()]
    attrs.extend(clf_attrs)

    # Deduplicate while preserving order
    seen = set()
    unique_attrs = []
    for a in attrs:
        if a not in seen:
            seen.add(a)
            unique_attrs.append(a)

    return unique_attrs[:max_attrs]

def attributes_to_caption(attrs):
    """Use a small LLM to turn attributes into a single natural sentence."""
    if not attrs:
        return "A photo of an animal."
    prompt = (
        "You are describing images very briefly. "
        "Write one short, natural English sentence that describes an image "
        "with the following visual attributes: " + .join(attrs) + "."
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=25,
            num_beams=3,
            do_sample=False,
        )
    caption = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    return caption

# 6) Walk the cats/dogs dataset and build captions
IMAGE_ROOT = Path("data/cats_dogs")  # same root as earlier in the notebook
image_paths = sorted(
    p for p in IMAGE_ROOT.rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

print(f"[run] Found {len(image_paths)} images under {IMAGE_ROOT.resolve()}.")

results = []
for img_path in image_paths:
    attrs = extract_image_attributes(img_path)
    caption = attributes_to_caption(attrs)
    results.append(
        {
            "image_path": str(img_path),
            "attributes": attrs,
            "caption": caption,
        }
    )
    print(f"{img_path}: {caption}")

# 7) Save to JSON and CSV for later use
json_path = IMAGE_ROOT / "captions_from_attributes.json"
csv_path = IMAGE_ROOT / "captions_from_attributes.csv"

with open(json_path, "w") as f:
    _json.dump(results, f, indent=2)
print(f"\n[save] JSON annotations saved to {json_path}")

with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["image_path", "caption", "attributes"])
    for r in results:
        writer.writerow([r["image_path"], r["caption"], ";".join(r["attributes"])] )
print(f"[save] CSV annotations saved to {csv_path}")
